 # Section 1: Setup, Unified Paths & Global Settings

 This cell loads essential utilities, configures global Pandas display parameters,

 silences silent-downcasting warnings, and dynamically maps project directories.

In [1]:
%load_ext autoreload
%autoreload 2

import ast

from pathlib import Path
import numpy as np
import pandas as pd

from core.contracts import EngineInput, FilterPack
from core.settings import TradingConfig
from core.utils import SystemUtils as SU
from data_pipeline import generate_features
from walk_forward import AlphaEngine, create_walk_forward_analyzer, run_headless_simulation

# 1. Global Configuration Options
config = TradingConfig()

# 2. Silence downcasting warnings & configure wide DataFrame display options
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 3000)

# 3. Dynamically resolve all project directories once
try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path.cwd()

data_dir = current_dir.parent / "data"
processed_dir = data_dir / "processed"
output_dir = current_dir.parent / "output"

print(f"Paths Resolved:\n - Root:      {current_dir}\n - Data:      {data_dir}\n - Processed: {processed_dir}\n - Output:    {output_dir}")




NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

Paths Resolved:
 - Root:      c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
 - Data:      c:\Users\ping\Files_win10\python\py311\stocks\data
 - Processed: c:\Users\ping\Files_win10\python\py311\stocks\data\processed
 - Output:    c:\Users\ping\Files_win10\python\py311\stocks\output


 # Section 2: Ingest Raw and Processed Datasets

 Safely loads raw inputs and previously stored processed matrices with existence verifications.

In [2]:
# Define Raw File Paths
df_ohlcv_path = data_dir / "df_OHLCV_stocks_etfs.parquet"
df_fed_path = data_dir / "High_Yield_Spread_T10Y2Y_Spread.csv"
df_indices_path = data_dir / "df_indices.parquet"

# Load Raw Files
if df_ohlcv_path.exists():
    df_ohlcv = pd.read_parquet(df_ohlcv_path)
    print("[OK] Successfully loaded df_ohlcv!")
else:
    print(f"[ERROR] File not found: {df_ohlcv_path}")

if df_fed_path.exists():
    df_fed = pd.read_csv(df_fed_path)
    print("[OK] Successfully loaded df_fed!")
else:
    print(f"[ERROR] File not found: {df_fed_path}")

if df_indices_path.exists():
    df_indices = pd.read_parquet(df_indices_path)
    print("[OK] Successfully loaded df_indices!")
else:
    print(f"[ERROR] File not found: {df_indices_path}")

[OK] Successfully loaded df_ohlcv!
[OK] Successfully loaded df_fed!
[OK] Successfully loaded df_indices!


In [3]:
# Define Processed File Paths
features_df_path = processed_dir / "features_df.parquet"
macro_df_path = processed_dir / "macro_df.parquet"
df_close_wide_path = processed_dir / "df_close_wide.parquet"
df_atrp_wide_path = processed_dir / "df_atrp_wide.parquet"
df_trp_wide_path = processed_dir / "df_trp_wide.parquet"

# Load Processed Files
if features_df_path.exists():
    features_df = pd.read_parquet(features_df_path)
    print(f"[OK] Loaded features_df  | Shape: {features_df.shape}")
else:
    print(f"[ERROR] File not found: {features_df_path}")

if macro_df_path.exists():
    macro_df = pd.read_parquet(macro_df_path)
    print(f"[OK] Loaded macro_df     | Shape: {macro_df.shape}")
else:
    print(f"[ERROR] File not found: {macro_df_path}")

if df_close_wide_path.exists():
    df_close_wide = pd.read_parquet(df_close_wide_path)
    print(f"[OK] Loaded df_close_wide| Shape: {df_close_wide.shape}")
else:
    print(f"[ERROR] File not found: {df_close_wide_path}")

if df_atrp_wide_path.exists():
    df_atrp_wide = pd.read_parquet(df_atrp_wide_path)
    print(f"[OK] Loaded df_atrp_wide | Shape: {df_atrp_wide.shape}")
else:
    print(f"[ERROR] File not found: {df_atrp_wide_path}")

if df_trp_wide_path.exists():
    df_trp_wide = pd.read_parquet(df_trp_wide_path)
    print(f"[OK] Loaded df_trp_wide  | Shape: {df_trp_wide.shape}")
else:
    print(f"[ERROR] File not found: {df_trp_wide_path}")

[OK] Loaded features_df  | Shape: (9560692, 18)
[OK] Loaded macro_df     | Shape: (16220, 11)
[OK] Loaded df_close_wide| Shape: (16220, 1577)
[OK] Loaded df_atrp_wide | Shape: (16220, 1577)
[OK] Loaded df_trp_wide  | Shape: (16220, 1577)


 # Section 3: In-Memory Structural Previews

 Outputs head and tail previews of the core dataframes.

In [4]:
print("\n" + "=" * 50)
print("DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)")
print("=" * 50)

# 1. Preview wide matrix prices
if "df_close_wide" in locals():
    print("\n=== df_close_wide (Prices) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(df_close_wide.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(df_close_wide.tail(3))

# 2. Preview multi-index features
if "features_df" in locals():
    print("\n=== features_df (Multi-Index Features) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(features_df.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(features_df.tail(3))

# 3. Preview macro data
if "macro_df" in locals():
    print("\n=== macro_df (Macro Data) ===")
    print("--- HEAD (First 3 Rows) ---")
    display(macro_df.head(3))
    print("\n--- TAIL (Last 3 Rows) ---")
    display(macro_df.tail(3))


DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)

=== df_close_wide (Prices) ===
--- HEAD (First 3 Rows) ---


Ticker,A,AA,AAL,AAON,AAPL,ABBV,ABEV,ABNB,ABT,ABVX,ACGL,ACI,ACM,ACN,ACWI,ACWX,ADBE,ADC,ADI,ADM,ADP,ADSK,ADT,AEE,AEG,AEIS,AEM,AEP,AER,AES,AFG,AFL,AFRM,AG,AGCO,AGG,AGI,AGNC,AHR,AIG,AIQ,AIRR,AIT,AIZ,AJG,AKAM,AKRE,ALAB,ALB,ALC,ALGN,ALL,ALLE,ALLY,ALNY,ALSN,ALV,AM,AMAT,AMCR,AMD,AME,AMG,AMGN,AMH,AMKR,AMLP,AMP,AMRZ,AMT,AMTM,AMX,AMZN,AN,ANET,AON,AOS,APA,APD,APG,APH,APLD,APO,APP,APPF,APTV,AR,ARCC,ARE,ARES,ARGX,ARKK,ARM,ARMK,ARWR,AS,ASML,ASND,ASR,ASTS,ASX,ATI,ATO,ATR,AU,AUR,AVAV,AVB,AVDE,AVDV,AVEM,AVGO,AVLV,AVTR,AVUS,AVUV,AVY,AWI,AWK,AXIA,AXON,AXP,AXS,AXSM,AXTA,AYI,AZN,AZO,B,BA,BABA,BAC,BAH,BAI,BALL,BAM,BAP,BAX,BBAX,BBCA,BBD,BBEU,BBIN,BBIO,BBJP,BBUS,BBVA,BBY,BCE,BCH,BCS,BDX,BE,BEKE,BEN,BEP,BEPC,BETA,BF-A,BF-B,BG,BHP,BIDU,BIIB,BIL,BILI,BINC,BIO,BIP,BIRK,BIV,BJ,BK,BKLC,BKLN,BKNG,BKR,BLD,BLDR,BLK,BLSH,BLV,BMNR,BMO,BMRN,BMY,BN,BND,BNDX,BNO,BNS,BNT,BNTX,BOKF,BOND,BOXX,BP,BPOP,BR,BRK-A,BRK-B,BRKR,BRO,BROS,BRX,BSAC,BSBR,BSCQ,BSCR,BSOL,BSV,BSX,BSY,BTC,BTI,BUD,BUFR,BURL,BVN,BWA,BWXT,BX,BXP,BYD,BZ,C,CACI,CAE,CAG,CAH,CAI,CARR,CART,CASY,CAT,CB,CBOE,CBRE,CBSH,CCEP,CCI,CCJ,CCK,CCL,CDE,CDNS,CDP,CDW,CEF,CEG,CELH,CF,CFG,CFR,CG,CGBL,CGCP,CGDV,CGGO,CGGR,CGMU,CGUS,CGXU,CHD,CHDN,CHKP,CHRW,CHT,CHTR,CHWY,CHYM,CI,CIB,CIBR,CIEN,CIFR,CIGI,CINF,CL,CLF,CLH,CLS,CLX,CM,CMC,CMCSA,CME,CMG,CMI,CMS,CNA,CNC,CNH,CNI,CNM,CNP,CNQ,COF,COHR,COIN,COKE,COLB,COO,COP,COPX,COR,CORT,COST,COWZ,CP,CPAY,CPB,CPNG,CPRT,CPT,CQP,CR,CRBG,CRCL,CRDO,CRH,CRK,CRL,CRM,CRS,CRWD,CRWV,CSCO,CSGP,CSL,CSX,CTAS,CTRE,CTSH,CTVA,CUBE,CVE,CVNA,CVS,CVX,CW,CWB,CWEN,CWEN-A,CX,CYTK,D,DAL,DASH,DB,DBEF,DBX,DCI,DD,DDOG,DDS,DE,DECK,DELL,DEO,DFAC,DFAE,DFAI,DFAS,DFAT,DFAU,DFAX,DFCF,DFEM,DFIC,DFIS,DFIV,DFLV,DFSD,DFSV,DFUS,DFUV,DG,DGRO,DGRW,DGX,DHI,DHR,DIA,DIHP,DINO,DIS,DIVO,DKNG,DKS,DLN,DLR,DLTR,DOC,DOCS,DOCU,DON,DOV,DOW,DOX,DPZ,DRI,DRS,DSGX,DSI,DT,DTE,DTM,DUHP,DUK,DUOL,DVA,DVN,DVY,DXCM,DXJ,DY,DYNF,E,EA,EAGG,EBAY,EC,ECL,ED,EDU,EDV,EEM,EFA,EFAV,EFG,EFV,EFX,EG,EGO,EGP,EHC,EIX,EL,ELAN,ELS,ELV,EMA,EMB,EMBJ,EME,EMLC,EMN,EMR,EMXC,ENB,ENSG,ENTG,EOG,EPAM,EPD,EQH,EQIX,EQNR,EQR,EQT,EQX,ERIC,ERIE,ES,ESAB,ESGD,ESGE,ESGU,ESGV,ESLT,ESS,ESTC,ET,ETHA,ETHB,ETN,ETR,EUFN,EVR,EVRG,EVTR,EW,EWBC,EWJ,EWT,EWY,EWZ,EXC,EXE,EXEL,EXLS,EXP,EXPD,EXPE,EXR,EZU,F,FAF,FANG,FAST,FBCG,FBND,FBTC,FCFS,FCNCA,FCX,FDL,FDN,FDS,FDVV,FDX,FE,FELC,FELG,FENI,FER,FERG,FEZ,FFIV,FHN,FICO,FIG,FIGR,FIS,FISV,FITB,FIVE,FIX,FLEX,FLOT,FLR,FLS,FLUT,FMDE,FMS,FMX,FN,FND,FNDA,FNDE,FNDF,FNDX,FNF,FNV,FOX,FOXA,FPE,FR,FRHC,FRMI,FROG,FRT,FSEC,FSLR,FSS,FSV,FTAI,FTCS,FTEC,FTI,FTNT,FTS,FTSM,FTV,FUTU,FVD,FWONA,FWONK,FXI,G,GAP,GBIL,GBTC,GD,GDDY,GDS,GDX,GDXJ,GE,GEHC,GEN,GEV,GFI,GFL,GFS,GGAL,GGG,GH,GIB,GIL,GILD,GIS,GL,GLBE,GLD,GLDM,GLPI,GLW,GM,GMAB,GME,GMED,GNRC,GOOG,GOOGL,GOVT,GPC,GPN,GRAB,GRID,GRMN,GS,GSAT,GSIE,GSK,GSLC,GTLB,GTLS,GUNR,GWRE,GWW,H,HAL,HALO,HAS,HBAN,HBM,HCA,HD,HDB,HDV,HEFA,HEI,HEI-A,HESM,HIG,HII,HIMS,HL,HLI,HLN,HLNE,HLT,HMC,HMY,HON,HOOD,HPE,HPQ,HQY,HRL,HSBC,HSIC,HST,HSY,HTHT,HUBB,HUBS,HUM,HWM,HYG,IAG,IAGG,IAU,IAUM,IBB,IBIT,IBKR,IBM,IBN,IBP,ICE,ICL,ICLR,ICSH,IDA,IDCC,IDEV,IDV,IDXX,IEF,IEFA,IEI,IEMG,IESC,IEUR,IEX,IFF,IGF,IGIB,IGM,IGSB,IGV,IHG,IHI,IJH,IJJ,IJK,IJR,IJS,IJT,ILMN,IMO,INCY,INDA,INFY,ING,INGR,INSM,INTC,INTU,INVH,IONQ,IONS,IOO,IOT,IP,IQLT,IQV,IR,IREN,IRM,ISRG,ISTB,IT,ITA,ITOT,ITT,ITUB,ITW,IUSB,IUSG,IUSV,IVE,IVV,IVW,IVZ,IWB,IWD,IWF,IWM,IWN,IWO,IWP,IWR,IWS,IWV,IWY,IX,IXJ,IXN,IXUS,IYF,IYR,IYW,J,JAAA,JAVA,JAZZ,JBHT,JBL,JBND,JBS,JBTM,JCI,JCPB,JD,JEF,JEPI,JEPQ,JGLO,JGRO,JHG,JHMM,JHX,JIRE,JKHY,JLL,JMBS,JMST,JMTG,JMUB,JNJ,JNK,JOBY,JPIE,JPM,JPST,JQUA,JXN,KB,KBWB,KDP,KEP,KEY,KEYS,KGC,KHC,KIM,KKR,KLAC,KLAR,KMB,KMI,KNSL,KNX,KO,KR,KRE,KRMN,KRYS,KSPI,KT,KTOS,KVUE,KVYO,KWEB,KYMR,L,LAD,LAMR,LBRDA,LBRDK,LDOS,LECO,LEN,LEVI,LFUS,LH,LHX,LI,LII,LIN,LINE,LITE,LKQ,LLY,LLYVA,LLYVK,LMBS,LMT,LNC,LNG,LNT,LOGI,LOW,LPLA,LQD,LRCX,LSCC,LTM,LULU,LUMN,LUV,LVS,LW,LYB,LYFT,LYG,LYV,MA,MAA,MAGS,MANH,MAR,MAS,MASI,MBB,MBLY,MCD,MCHI,MCHP,MCK,MCO,MDB,MDGL,MDLZ,MDT,MDY,MEDP,MELI,MET,META,MFC,MFG,MGA,MGC,MGK,MGM,MGV,MHK,MICC,MIDD,MINT,MKC,MKL,MKSI,MKTX,MLI,MLM,MMM,MMYT,MNDY,MNST,MO,MOAT,MOD,MOG-A,MOH,MORN,MO


--- TAIL (Last 3 Rows) ---


Ticker,A,AA,AAL,AAON,AAPL,ABBV,ABEV,ABNB,ABT,ABVX,ACGL,ACI,ACM,ACN,ACWI,ACWX,ADBE,ADC,ADI,ADM,ADP,ADSK,ADT,AEE,AEG,AEIS,AEM,AEP,AER,AES,AFG,AFL,AFRM,AG,AGCO,AGG,AGI,AGNC,AHR,AIG,AIQ,AIRR,AIT,AIZ,AJG,AKAM,AKRE,ALAB,ALB,ALC,ALGN,ALL,ALLE,ALLY,ALNY,ALSN,ALV,AM,AMAT,AMCR,AMD,AME,AMG,AMGN,AMH,AMKR,AMLP,AMP,AMRZ,AMT,AMTM,AMX,AMZN,AN,ANET,AON,AOS,APA,APD,APG,APH,APLD,APO,APP,APPF,APTV,AR,ARCC,ARE,ARES,ARGX,ARKK,ARM,ARMK,ARWR,AS,ASML,ASND,ASR,ASTS,ASX,ATI,ATO,ATR,AU,AUR,AVAV,AVB,AVDE,AVDV,AVEM,AVGO,AVLV,AVTR,AVUS,AVUV,AVY,AWI,AWK,AXIA,AXON,AXP,AXS,AXSM,AXTA,AYI,AZN,AZO,B,BA,BABA,BAC,BAH,BAI,BALL,BAM,BAP,BAX,BBAX,BBCA,BBD,BBEU,BBIN,BBIO,BBJP,BBUS,BBVA,BBY,BCE,BCH,BCS,BDX,BE,BEKE,BEN,BEP,BEPC,BETA,BF-A,BF-B,BG,BHP,BIDU,BIIB,BIL,BILI,BINC,BIO,BIP,BIRK,BIV,BJ,BK,BKLC,BKLN,BKNG,BKR,BLD,BLDR,BLK,BLSH,BLV,BMNR,BMO,BMRN,BMY,BN,BND,BNDX,BNO,BNS,BNT,BNTX,BOKF,BOND,BOXX,BP,BPOP,BR,BRK-A,BRK-B,BRKR,BRO,BROS,BRX,BSAC,BSBR,BSCQ,BSCR,BSOL,BSV,BSX,BSY,BTC,BTI,BUD,BUFR,BURL,BVN,BWA,BWXT,BX,BXP,BYD,BZ,C,CACI,CAE,CAG,CAH,CAI,CARR,CART,CASY,CAT,CB,CBOE,CBRE,CBSH,CCEP,CCI,CCJ,CCK,CCL,CDE,CDNS,CDP,CDW,CEF,CEG,CELH,CF,CFG,CFR,CG,CGBL,CGCP,CGDV,CGGO,CGGR,CGMU,CGUS,CGXU,CHD,CHDN,CHKP,CHRW,CHT,CHTR,CHWY,CHYM,CI,CIB,CIBR,CIEN,CIFR,CIGI,CINF,CL,CLF,CLH,CLS,CLX,CM,CMC,CMCSA,CME,CMG,CMI,CMS,CNA,CNC,CNH,CNI,CNM,CNP,CNQ,COF,COHR,COIN,COKE,COLB,COO,COP,COPX,COR,CORT,COST,COWZ,CP,CPAY,CPB,CPNG,CPRT,CPT,CQP,CR,CRBG,CRCL,CRDO,CRH,CRK,CRL,CRM,CRS,CRWD,CRWV,CSCO,CSGP,CSL,CSX,CTAS,CTRE,CTSH,CTVA,CUBE,CVE,CVNA,CVS,CVX,CW,CWB,CWEN,CWEN-A,CX,CYTK,D,DAL,DASH,DB,DBEF,DBX,DCI,DD,DDOG,DDS,DE,DECK,DELL,DEO,DFAC,DFAE,DFAI,DFAS,DFAT,DFAU,DFAX,DFCF,DFEM,DFIC,DFIS,DFIV,DFLV,DFSD,DFSV,DFUS,DFUV,DG,DGRO,DGRW,DGX,DHI,DHR,DIA,DIHP,DINO,DIS,DIVO,DKNG,DKS,DLN,DLR,DLTR,DOC,DOCS,DOCU,DON,DOV,DOW,DOX,DPZ,DRI,DRS,DSGX,DSI,DT,DTE,DTM,DUHP,DUK,DUOL,DVA,DVN,DVY,DXCM,DXJ,DY,DYNF,E,EA,EAGG,EBAY,EC,ECL,ED,EDU,EDV,EEM,EFA,EFAV,EFG,EFV,EFX,EG,EGO,EGP,EHC,EIX,EL,ELAN,ELS,ELV,EMA,EMB,EMBJ,EME,EMLC,EMN,EMR,EMXC,ENB,ENSG,ENTG,EOG,EPAM,EPD,EQH,EQIX,EQNR,EQR,EQT,EQX,ERIC,ERIE,ES,ESAB,ESGD,ESGE,ESGU,ESGV,ESLT,ESS,ESTC,ET,ETHA,ETHB,ETN,ETR,EUFN,EVR,EVRG,EVTR,EW,EWBC,EWJ,EWT,EWY,EWZ,EXC,EXE,EXEL,EXLS,EXP,EXPD,EXPE,EXR,EZU,F,FAF,FANG,FAST,FBCG,FBND,FBTC,FCFS,FCNCA,FCX,FDL,FDN,FDS,FDVV,FDX,FE,FELC,FELG,FENI,FER,FERG,FEZ,FFIV,FHN,FICO,FIG,FIGR,FIS,FISV,FITB,FIVE,FIX,FLEX,FLOT,FLR,FLS,FLUT,FMDE,FMS,FMX,FN,FND,FNDA,FNDE,FNDF,FNDX,FNF,FNV,FOX,FOXA,FPE,FR,FRHC,FRMI,FROG,FRT,FSEC,FSLR,FSS,FSV,FTAI,FTCS,FTEC,FTI,FTNT,FTS,FTSM,FTV,FUTU,FVD,FWONA,FWONK,FXI,G,GAP,GBIL,GBTC,GD,GDDY,GDS,GDX,GDXJ,GE,GEHC,GEN,GEV,GFI,GFL,GFS,GGAL,GGG,GH,GIB,GIL,GILD,GIS,GL,GLBE,GLD,GLDM,GLPI,GLW,GM,GMAB,GME,GMED,GNRC,GOOG,GOOGL,GOVT,GPC,GPN,GRAB,GRID,GRMN,GS,GSAT,GSIE,GSK,GSLC,GTLB,GTLS,GUNR,GWRE,GWW,H,HAL,HALO,HAS,HBAN,HBM,HCA,HD,HDB,HDV,HEFA,HEI,HEI-A,HESM,HIG,HII,HIMS,HL,HLI,HLN,HLNE,HLT,HMC,HMY,HON,HOOD,HPE,HPQ,HQY,HRL,HSBC,HSIC,HST,HSY,HTHT,HUBB,HUBS,HUM,HWM,HYG,IAG,IAGG,IAU,IAUM,IBB,IBIT,IBKR,IBM,IBN,IBP,ICE,ICL,ICLR,ICSH,IDA,IDCC,IDEV,IDV,IDXX,IEF,IEFA,IEI,IEMG,IESC,IEUR,IEX,IFF,IGF,IGIB,IGM,IGSB,IGV,IHG,IHI,IJH,IJJ,IJK,IJR,IJS,IJT,ILMN,IMO,INCY,INDA,INFY,ING,INGR,INSM,INTC,INTU,INVH,IONQ,IONS,IOO,IOT,IP,IQLT,IQV,IR,IREN,IRM,ISRG,ISTB,IT,ITA,ITOT,ITT,ITUB,ITW,IUSB,IUSG,IUSV,IVE,IVV,IVW,IVZ,IWB,IWD,IWF,IWM,IWN,IWO,IWP,IWR,IWS,IWV,IWY,IX,IXJ,IXN,IXUS,IYF,IYR,IYW,J,JAAA,JAVA,JAZZ,JBHT,JBL,JBND,JBS,JBTM,JCI,JCPB,JD,JEF,JEPI,JEPQ,JGLO,JGRO,JHG,JHMM,JHX,JIRE,JKHY,JLL,JMBS,JMST,JMTG,JMUB,JNJ,JNK,JOBY,JPIE,JPM,JPST,JQUA,JXN,KB,KBWB,KDP,KEP,KEY,KEYS,KGC,KHC,KIM,KKR,KLAC,KLAR,KMB,KMI,KNSL,KNX,KO,KR,KRE,KRMN,KRYS,KSPI,KT,KTOS,KVUE,KVYO,KWEB,KYMR,L,LAD,LAMR,LBRDA,LBRDK,LDOS,LECO,LEN,LEVI,LFUS,LH,LHX,LI,LII,LIN,LINE,LITE,LKQ,LLY,LLYVA,LLYVK,LMBS,LMT,LNC,LNG,LNT,LOGI,LOW,LPLA,LQD,LRCX,LSCC,LTM,LULU,LUMN,LUV,LVS,LW,LYB,LYFT,LYG,LYV,MA,MAA,MAGS,MANH,MAR,MAS,MASI,MBB,MBLY,MCD,MCHI,MCHP,MCK,MCO,MDB,MDGL,MDLZ,MDT,MDY,MEDP,MELI,MET,META,MFC,MFG,MGA,MGC,MGK,MGM,MGV,MHK,MICC,MIDD,MINT,MKC,MKL,MKSI,MKTX,MLI,MLM,MMM,MMYT,MNDY,MNST,MO,MOAT,MOD,MOG-A,MOH,MORN,MO


=== features_df (Multi-Index Features) ===
--- HEAD (First 3 Rows) ---


...

ATR      ATRP       TRP        RSI    Mom_21  Consistency     IR_63   Beta_63     DD_21  AutoCorr_15    Ret_1d  Range_Pos_20  Slope_P_5  Slope_V_5  Convexity  RollingStalePct  RollMedDollarVol  RollingSameVolCount
Ticker Date                                                                                                                                                                                                                                  
ZWS    2026-06-10  1.241188  0.026341  0.036290  44.143181 -0.074245          0.4 -0.047516  1.207976 -0.050993    -0.427133 -0.024834      0.342616   0.000590  -0.220716   0.001688              0.0      3.885449e+07                  0.0
       2026-06-11  1.263961  0.026261  0.032412  50.205407 -0.030651          0.4 -0.035968  1.198699 -0.028701    -0.335414  0.021435      0.604354   0.002971   0.028204  -0.000261              0.0      3.885449e+07                  0.0
       2026-06-12  1.223678  0.025288  0.014466  51.659863 -0.006655          0.6 -0.038373  1.205290 -0.023454    -0.286505  0.005402      0.671505   0.004501   0.269625   0.003911              0.0      3.885449e+07                  0.0


=== macro_df (Macro Data) ===
--- HEAD (First 3 Rows) ---


,Mkt_Ret,Macro_Trend,High_Yield_Spread,Yield_Curve_10Y2Y,High_Yield_Spread_Z,Yield_Curve_10Y2Y_Z,Macro_Trend_Vel,Macro_Trend_Vel_Z,Macro_Trend_Mom,Macro_Vix_Z,Macro_Vix_Ratio
Date,,,,,,,,,,,
1962-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1962-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1962-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0



--- TAIL (Last 3 Rows) ---


,Mkt_Ret,Macro_Trend,High_Yield_Spread,Yield_Curve_10Y2Y,High_Yield_Spread_Z,Yield_Curve_10Y2Y_Z,Macro_Trend_Vel,Macro_Trend_Vel_Z,Macro_Trend_Mom,Macro_Vix_Z,Macro_Vix_Ratio
Date,,,,,,,,,,,
2026-06-10,-0.015766,0.062031,0.0,0.0,0.0,0.0,-0.039580,-0.874239,-0.039580,0.485803,0.970730
2026-06-11,0.016997,0.079287,0.0,0.0,0.0,0.0,-0.019784,-0.439408,-0.019784,-0.176384,0.907563
2026-06-12,0.005408,0.084315,0.0,0.0,0.0,0.0,-0.019995,-0.447505,-0.019995,-0.599436,0.862019


 # Section 4: Engine Initialization & Stage 1 Analysis

In [5]:
# Instantiate Master Engine
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("Master AlphaEngine Ready.")

Master AlphaEngine Ready.


In [ ]:
# Configuration for Interactive UI (Stage 1 Setup)
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=189,
    holding_period=5,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=200,
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [7]:
# Execute Headless Agent Simulation
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df.style.format("{:.4f}"))
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
----------------------------------------------------------------------
Timeline: [2025-07-16] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (200):
SGOV, SHV, BIL, MINT, USFR, BOXX, PULS, JPST, GTLS, SNDK
JAAA, RVMD, SATS, CIEN, TER, EA, LITE, ASX, VGSH, WDC
GLW, ROIV, SHY, ROST, EWY, WBD, VALE, MU, FTI, ERIC
RIO, VTEB, BE, KEYS, NOK, STX, FIX, UTHR, VCSH, IGSB
MUB, CHRW, EMXC, LRCX, GOOGL, GOOG, HSBC, INTC, TEVA, JNJ
TD, CAT, EWZ, ALB, MKSI, CMI, BNS, BSV, COPX, RY
SU, PBR, EFV, ASML, GSK, EWT, FDX, ONTO, BP, AA
IAG, CVE, SLV, HL, KGC, ITUB, COHR, BHP, AU, JAZZ
IAUM, AMAT, GLDM, SGOL, IAU, APA, B, AZN, XBI, SCCO
EMB, PSLV, EQX, SOXX, SCHF, GLD, VEA, BG, SPDW, NEM
IEMG, VEU, IONS, GDX, EEM, VXUS, PHYS, LSCC, HAL, IXUS
ETR, EWJ, MTZ, OPEN, SMH, FNDX, USO, SOXL, KLAC, GDXJ
MOD, NBIS, FTAI, SIL, MEDP, VLO, MPWR, BK, SPIB, VRT
SHEL, AEM, MBB, TTMI, JBHT, BKR, CNQ, IDEV, DD, GFI
ATI, WELL, FE, JNK, EQIX, NVS, FNV, MTSI, VTR,

,Full,Lookback,Holding
Metric,,,
Group Gain,0.6884,0.6641,0.0028
Benchmark Gain,0.1427,0.1254,0.0053
== Gain Delta,0.5457,0.5387,-0.0026
Group Sharpe,4.0505,3.9996,1.2023
Benchmark Sharpe,1.5411,1.4013,2.3401
== Sharpe Delta,2.5094,2.5983,-1.1378
Group Sharpe (ATRP),0.1108,0.1109,0.0199
Benchmark Sharpe (ATRP),0.0723,0.0662,0.0879
== Sharpe (ATRP) Delta,0.0385,0.0447,-0.0680



Target Reward Signal: 0.002771


 # Section 5: Stage 2 - Cascade Filter Execution

In [8]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# LAUNCH STAGE 2 (Cascade on the Stage 1 subset)
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [9]:
# Parse and map results from Analyzer 2
result = SU.map_analyzer(analyzer2)

[SEARCH] ANALYZER SIMULATION MAP
[  0] [DATA] audit_pack (EngineOutput)
[  1]   [PLOT] portfolio_series (shape=(196,))
[  2]   [PLOT] benchmark_series (shape=(196,))
[  3]   [CALC] normalized_plot_data (shape=(196, 100))
[  4]   [FILE] tickers (len=100)
[  5]     [DOC] index_0 (str)
[  6]     [DOC] index_1 (str)
[  7]     [DOC] index_2 (str)
[  8]     [DOC] index_3 (str)
[  9]     [DOC] index_4 (str)
[ 10]     [DOC] index_5 (str)
[ 11]     [DOC] index_6 (str)
[ 12]     [DOC] index_7 (str)
[ 13]     [DOC] index_8 (str)
[ 14]     [DOC] index_9 (str)
[ 15]     [DOC] index_10 (str)
[ 16]     [DOC] index_11 (str)
[ 17]     [DOC] index_12 (str)
[ 18]     [DOC] index_13 (str)
[ 19]     [DOC] index_14 (str)
[ 20]     [DOC] index_15 (str)
[ 21]     [DOC] index_16 (str)
[ 22]     [DOC] index_17 (str)
[ 23]     [DOC] index_18 (str)
[ 24]     [DOC] index_19 (str)
[ 25]     [DOC] index_20 (str)
[ 26]     [DOC] index_21 (str)
[ 27]     [DOC] index_22 (str)
[ 28]     [DOC] index_23 (str)
[ 29]     [D

In [10]:
# Test different methods to extract resulting target tickers
_tickers = SU.fetch(4, result)
print(f"_tickers (by index): {_tickers}\n")

_tickers = SU.fetch("tickers", result)
print(f"_tickers (by key): {_tickers}\n")

print(f'result[4]["obj"]: {result[4]["obj"]}')

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'SATS',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM']

_tickers (by index): ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'SATS', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', 'VTEB', 'BE', 'KEYS', 'NOK', 'STX', 'FIX', 'UTHR', 'VCSH', 'IGSB', 'MUB', 'CHRW', 'EMXC', 'LRCX', 'GOOGL', 'GOOG', 'HSBC', 'INTC', 'TEVA', 'JNJ', 'TD', 'CAT', 'EWZ', 'ALB', 'MKSI', 'CMI', 'BNS', 'BSV', 'COPX', 'RY', 'SU', 'PBR', 'EFV', 'ASML', 'GSK', 'EWT', 'FDX', 'ONTO', 'BP', 'AA', 'IAG', 'CVE', 'SLV', 'HL', 'KGC', 'ITUB', 'COHR', 'BHP', 'AU', 'JAZZ', 'IAUM', 'AMAT', 'GLDM', 'SGOL', 'IAU', 'APA', 'B', 'AZN', 'XBI', 'SCCO', 'EMB', 'PSLV', 'EQX', 'SOXX', 'SCHF', 'GLD', 'VEA', 'BG', 'SPDW', 'NEM']

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'SATS',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM']

_tickers (by key): ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'SATS', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC', 'RIO', 'VTEB', 'BE', 'KEYS', 'NOK', 'STX', 'FIX', 'UTHR', 'VCSH', 'IGSB', 'MUB', 'CHRW', 'EMXC', 'LRCX', 'GOOGL', 'GOOG', 'HSBC', 'INTC', 'TEVA', 'JNJ', 'TD', 'CAT', 'EWZ', 'ALB', 'MKSI', 'CMI', 'BNS', 'BSV', 'COPX', 'RY', 'SU', 'PBR', 'EFV', 'ASML', 'GSK', 'EWT', 'FDX', 'ONTO', 'BP', 'AA', 'IAG', 'CVE', 'SLV', 'HL', 'KGC', 'ITUB', 'COHR', 'BHP', 'AU', 'JAZZ', 'IAUM', 'AMAT', 'GLDM', 'SGOL', 'IAU', 'APA', 'B', 'AZN', 'XBI', 'SCCO', 'EMB', 'PSLV', 'EQX', 'SOXX', 'SCHF', 'GLD', 'VEA', 'BG', 'SPDW', 'NEM']

result[4]["obj"]: ['SGOV', 'SHV', 'BIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'GTLS', 'SNDK', 'JAAA', 'RVMD', 'SATS', 'CIEN', 'TER', 'EA', 'LITE', 'ASX', 'VGSH', 'WDC', 'GLW', 'ROIV', 'SHY', 'ROST', 'EWY', 'WBD', 'VALE', 'MU', 'FTI', 'ERIC',

 # Section 6: Verification, Exports & Auditing

In [11]:
# Export analytical trace logs to Excel
SU.export_audit_to_excel(
    audit_pack=analyzer2.last_run, filename="Audit_Verification_Report.xlsx"
)

[FILE] [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx
[DONE] Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/Audit_Verification_Report.xlsx')

In [12]:
# Export last-run stacked OHLCV ticker records
SU.export_last_run_tickers_data_to_csv(
    analyzer=analyzer2,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename="last_run_tickers_stacked.csv",
)

Creating combined dictionary for 101 ticker(s)
Date range: 2025-07-16 00:00:00 to 2026-04-24 00:00:00
Data retrieved for 101 ticker(s) from 2025-07-16 00:00:00 to 2026-04-24 00:00:00
Total rows: 19796
Date range in data: 2025-07-16 00:00:00 to 2026-04-24 00:00:00
  SGOV: 196 rows
  SHV: 196 rows
  BIL: 196 rows
  MINT: 196 rows
  USFR: 196 rows
  BOXX: 196 rows
  PULS: 196 rows
  JPST: 196 rows
  GTLS: 196 rows
  SNDK: 196 rows
  JAAA: 196 rows
  RVMD: 196 rows
  SATS: 196 rows
  CIEN: 196 rows
  TER: 196 rows
  EA: 196 rows
  LITE: 196 rows
  ASX: 196 rows
  VGSH: 196 rows
  WDC: 196 rows
  GLW: 196 rows
  ROIV: 196 rows
  SHY: 196 rows
  ROST: 196 rows
  EWY: 196 rows
  WBD: 196 rows
  VALE: 196 rows
  MU: 196 rows
  FTI: 196 rows
  ERIC: 196 rows
  RIO: 196 rows
  VTEB: 196 rows
  BE: 196 rows
  KEYS: 196 rows
  NOK: 196 rows
  STX: 196 rows
  FIX: 196 rows
  UTHR: 196 rows
  VCSH: 196 rows
  IGSB: 196 rows
  MUB: 196 rows
  CHRW: 196 rows
  EMXC: 196 rows
  LRCX: 196 rows
  GOOGL: 

WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/last_run_tickers_stacked.csv')

 # Section 7: Slicing Tests & Dictionary Parsing

In [13]:
# Verify captured ticker names
_tickers

['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'SATS',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM']

In [14]:
# Test various pandas multi-index slicing configurations
test_tickers = ["NVDA", "GOOG"]
idx = pd.IndexSlice
columns = ["Adj Close", "Volume"]

# Slice 1: All dates, all columns
display(df_ohlcv.loc[idx[test_tickers]].copy().head(3))

# Slice 2: specific start date, all columns
display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], :].head(3))

# Slice 3: specific start date, selected columns
display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], columns].head(3))

# Slice 4: specific start/end range, all columns
display(
    df_ohlcv.loc[
        idx[test_tickers, pd.Timestamp("2026-04-01") : pd.Timestamp("2026-04-10")], :
    ].head(3)
)

Adj Open  Adj High   Adj Low  Adj Close      Volume
Ticker Date                                                           
NVDA   1999-01-22  0.040063  0.044713  0.035532   0.037559  2964546624
       1999-01-25  0.040540  0.041970  0.037559   0.041494   557464553
       1999-01-26  0.041970  0.042805  0.037678   0.038274   374788019

Adj Open  Adj High  Adj Low  Adj Close     Volume
Ticker Date                                                         
NVDA   2026-04-01   175.795   177.164  174.547    175.545  168327949
       2026-04-02   171.980   177.283  171.170    177.183  143310037
       2026-04-06   176.954   177.583  175.555    177.433  107689668

Adj Close     Volume
Ticker Date                            
NVDA   2026-04-01    175.545  168327949
       2026-04-02    177.183  143310037
       2026-04-06    177.433  107689668

Adj Open  Adj High  Adj Low  Adj Close     Volume
Ticker Date                                                         
NVDA   2026-04-01   175.795   177.164  174.547    175.545  168327949
       2026-04-02   171.980   177.283  171.170    177.183  143310037
       2026-04-06   176.954   177.583  175.555    177.433  107689668

In [15]:
# Test custom combined dictionary extraction
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=_tickers,
    date_start="2026-05-01",
    date_end=None,
    verbose=False,
)

{'SGOV':             Adj Open  Adj High  Adj Low  Adj Close    Volume       ATR      ATRP       TRP        RSI    Mom_21  Consistency     IR_63   Beta_63  DD_21  AutoCorr_15    Ret_1d  Range_Pos_20  Slope_P_5  Slope_V_5  Convexity  RollingStalePct  RollMedDollarVol  RollingSameVolCount
 Date                                                                                                                                                                                                                                                                                 
 2026-05-01   100.112   100.112  100.102    100.112  33910416  0.019468  0.000194  0.000380  99.999829  0.003171          0.6 -0.061531 -0.003311    0.0    -0.574265  0.000380      1.000000   0.000146   0.739754   0.000006         0.011905      1.368668e+09                  0.0
 2026-05-04   100.122   100.122  100.112    100.122  20844209  0.018792  0.000188  0.000100  99.999837  0.002872          0.8 -0.047191 -0.003235    0.0   

 # Section 8: Finviz Ingestion & Filtering

In [16]:
# Load latest updated Finviz dataset dynamically
def load_latest_finviz_parquet(data_dir: Path) -> pd.DataFrame:
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"
    files = list(data_dir.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {data_dir.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Load the file using the unified data_dir path
df_finviz = load_latest_finviz_parquet(data_dir)
print(
    f"df_finviz raw records shape: {df_finviz.shape if df_finviz is not None else None}"
)

DEBUG: Loading file: 2026-06-12_df_finviz_merged_stocks_etfs.parquet
df_finviz raw records shape: (1434, 139)


In [17]:
# Set test target tickers
target_tickers = ["GLW", "NOK", "DELL", "NVDA", "GOOG"]

In [18]:
# Clean index extraction using ast exception parsing if tickers are missing
try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Strip out the missing symbols and retry slicing
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz].copy()
    result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

print(f"result shape: {result.shape}")

result shape: (5, 139)


In [19]:
# Display filtered Finviz information and schema details
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, GLW to GOOG
Columns: 139 entries, No. to Omega 250d
dtypes: float64(120), int64(3), object(16)
memory usage: 5.5+ KB


In [20]:
# Check specific target column (Market Capitalization)
result["MktCap AUM, M"]

GLW      154230.0
NOK       82770.0
DELL     256370.0
NVDA    4965600.0
GOOG    4349630.0
Name: MktCap AUM, M, dtype: float64

In [21]:
# Sort records descendingly by Market Capitalization
result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

In [22]:
# View final sorted DataFrame
print(result)

      No.                Company               Index                  Sector                        Industry  Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M    P/E  Fwd P/E   PEG    P/S    P/B    P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %   ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W Lo

In [23]:
# Export processed dataframe output directly to trash file
result.to_csv(output_dir / "_trash.csv")